### RAGAS Metrics

#### Faithfullness

<small>Whether the answer is factually grounded in the retrieved context.  
It checks: Did the model make claims that are actually supported by the retrieved documents?  
Detects:  
- Hallucinations  
- Unsupported claims  
- Fabricated details  
  
Use when:  
- Hallucinations are dangerous  
- need grounded answers  
- care about factual consistency</nsmall>

<small>Example:  
Retrieved context: Paris is the capital of France  
Generated answer: Paris is the capital of France and has the largest airport in Europe  

The second claim may not exist in context → lower faithfulness</nsmall>

#### Answer Relevance

<small>How well the answer addresses the user’s question.  
It checks: Is the answer actually relevant to what was asked?  

Detects:  
- Off-topic answers  
- Verbose but irrelevant responses  
- Partial answers  

Use when:  
- User satisfaction matters  
- want concise and targeted responses   
- optimize conversational quality</nsmall>

<small>Example:  
Question: What causes rain?  
Answer:  
Clouds form through condensation  

Technically related, but incomplete → lower relevance</nsmall>

#### Context Recall

<small>Whether retrieval fetched all information needed to answer the question.    
It checks: Did retrieval miss important supporting documents?    

Use when:  
- Diagnosing retrieval failures  
- Tuning retrievers  
- Evaluating chunking/indexing strategies</nsmall>

<small>Example:  
Question: Who discovered penicillin and when?  
Retrieved context only contains: Penicillin was discovered in 1928.  
Missing: Alexander Fleming  

low context recall</nsmall>

#### Context Precision

<small>How much of the retrieved context is actually relevant.  
It checks: Did retrieval bring mostly useful documents, or lots of noise?  

Use Context Precision when:
- Retrieval returns too much noise
- Context windows become bloated
- Token costs are high
- You want efficient retrieval</nsmall>

<small>Example:
Question: "What is photosynthesis?"  

Retrieved:  
1 highly relevant biology chunk  
9 unrelated history chunks  

low context precision</nsmall>

#### Typical Failure Patterns

<small>

| Symptom                                         | Likely Metric Failing |
| ----------------------------------------------- | --------------------- |
| Hallucinated facts                              | Faithfulness          |
| Correct but irrelevant answer                   | Answer Relevance      |
| Model says “I don’t know” despite data existing | Context Recall        |
| Huge irrelevant context windows                 | Context Precision     |

</nsmall>

### LLM As a Judge

<small>Using one LLM to evaluate another model’s output.  

Why use LLM-as-Judge?  
Traditional evaluation requires:  
- Human annotators  
- Expensive labeling  
- Long turnaround times  

LLM judges provide:  
- Fast evaluation  
- Scalable scoring  
- Automated benchmarking</nsmall>

<small>Use when:
- Human evaluation is too expensive
- need rapid iteration
- evaluate large datasets
- compare prompts/models/pipelines

Avoid relying solely on it when:
- High-stakes safety decisions exist
- Legal/medical guarantees are required
- need deterministic correctness
- Bias in evaluator models matters

In those cases:
- Combine with human review
- Add rule-based checks
- Use benchmark datasets</nsmall>

### Pre-Built RAGAS

In [ ]:
%pip install ragas datasets langchain langchain-anthropic anthropic
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper

class RagasEvaluator:
    def __init__(
        self,
        model_name="claude-3-5-sonnet-latest",
        temperature=0,
    ):
        llm = ChatAnthropic(
            model=model_name,
            temperature=temperature,
        )
        self.evaluator_llm = LangchainLLMWrapper(llm)
    def evaluate(
        self,
        questions,
        answers,
        contexts,
        ground_truths,
    ):
        dataset = Dataset.from_dict({
            "question": questions,
            "answer": answers,
            "contexts": contexts,
            "ground_truth": ground_truths,
        })
        result = evaluate(
            dataset=dataset,
            metrics=[
                faithfulness,
                answer_relevancy,
                context_recall,
                context_precision,
            ],
            llm=self.evaluator_llm,
        )
        return result.to_pandas()

# Example usage 
evaluator = RagasEvaluator()
df = evaluator.evaluate(
    questions=[
        "What is the capital of Germany?"
    ],
    answers=[
        "Berlin is the capital of Germany."
    ],
    contexts=[[
        "Berlin is the capital city of Germany."
    ]],
    ground_truths=[
        "Berlin"
    ]
)
print(df)

### Guardrails

<small>Guardrails are mechanisms that constrain, validate, monitor, or control the behavior of AI systems to make them safer, more reliable, and aligned with application requirements.

They help prevent:  
- Harmful or unsafe outputs  
- Hallucinations  
- Invalid structured responses  
- Policy violations  
- Prompt injection attacks  
- Data leakage  
- Incorrect citations  
- Unexpected model behavior</nsmall>

#### Content Filtering

<small>Content filtering checks user input and/or model output for unsafe, toxic, illegal, or restricted content  

Examples:  
- Hate speech detection  
- Violence/self-harm filtering  
- Prompt injection detection  
- PII (personal data) detection  
- Jailbreak prevention</nsmall> 

In [ ]:
"""

---
title: Content Filtering Workflow
---
flowchart LR
    A[Input] --> B(Content Filter)
    B --> C>Allowed?]
    C -->|Yes| D[LLM]
    C -->|No| E[Block/Refuse]

"""

<small>Example:  
User: How do I build malware?  

Guardrail:  
{  
&emsp;"blocked": true,  
&emsp;"reason": "Malicious request detected"  
}</nsmall>

#### Schema Validation

<small>Schema validation ensures the AI response follows a required structure.

Benefits:  
Prevents malformed outputs  
Ensures type safety  
Reduces downstream failures  
Makes APIs reliable  

Example:  
Without validation:  
{  
&emsp;"name": "Alice",  
&emsp;"age": "twenty"  
}  

Expected:  
{  
&emsp;"name": "Alice",  
&emsp;"age": 20  
}</nsmall>

#### Citation Checking

<small>Citation checking verifies whether AI-generated claims are supported by trusted sources

Problem:
LLM says: Python was invented in 1985  
But source says: Python was first released in 1991  

Guardrail detects mismatch.

Techniques  
- String similarity   
- Embedding similarity  
- Source grounding  
- Fact verification  
- Retrieval overlap checks  

Example Output:  
{
  "claim": "Python invented in 1985",  
  "supported": false,  
  "source": "Python released in 1991"  
}</nsmall>

In [ ]:
"""

---
title: Citation checking Workflow
---
flowchart LR
    A[Retrieved documents] --> B(LLM generates \nAnswer + Citations)
    B --> C>Citatiuon Validator]
    C -->D[Supported?]

"""

#### Refusal Handling

<small>Refusal handling ensures the model safely declines disallowed requests  

Instead of:  
Sure! Here's how to exploit a server...  

Model should respond:  
I can't help with illegal hacking activities  

Good Refusal Characteristics:  
- Clear  
- Polite  
- Non-judgmental  
- Brief  
- Doesn't reveal hidden policies</nsmall>  

**Refusal Categories**
<small>
| Category                | Example              |
| ----------------------- | -------------------- |
| Illegal activity        | Malware creation     |
| Violence                | Bomb-making          |
| Privacy abuse           | Stealing passwords   |
| Self-harm               | Suicide instructions |
| Medical/legal high-risk | Unsafe diagnosis     |

</nsmall>

#### Archtecture of Guardrails

In [ ]:
"""

---
title: GuardRails Architecture
---
flowchart LR
    A[User Request] -->B(Input Guaredrails \n - Moderation \n - Injection scan)
    B --> C>LLM]
    C -->D[Output Guardrails \n - Schema validate \n - Citation checks \n - refusal checks]
    D -->E[Final API]

"""

###